*Embeddings*

In [ ]:
#product embeddings#
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import torch

INPUT_CSV  = "data/cleaned_test.csv"
OUTPUT_CSV = "outputs/ccleaned_test_embeddings.csv"
MODEL_NAME = "intfloat/multilingual-e5-large-instruct"

# Load data
df = pd.read_csv(INPUT_CSV)
texts = df["cleaned_text"].fillna("").astype(str).tolist()

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)

# 👉 prefix as passage
texts_for_model = [f"passage: {t}" for t in texts]

# Embed
emb = model.encode(
    texts_for_model,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)
dim = emb.shape[1]

# Save
emb_cols = [f"v{i}" for i in range(1, dim + 1)]
out = pd.concat(
    [df.reset_index(drop=True), pd.DataFrame(emb, columns=emb_cols)], axis=1
)

if "row_id" not in out.columns:
    out.insert(0, "row_id", np.arange(1, len(out) + 1, dtype=np.int64))

out.to_csv(OUTPUT_CSV, index=False)


print(f"✅ Saved {len(out)} rows with {dim}-dim passage embeddings → {OUTPUT_CSV}")

*cosine similarity*

In [ ]:
import teradatasql
import pandas as pd
from config.settings import TD_HOST, TD_USER, TD_PASS, TD_DB

ITEM_EMB  = f"{TD_DB}.products_labels_fc"     # has: row_id, v1..vN
REF_EMB   = f"{TD_DB}.original_labels_fc"     # has: <some id>, v1..vN
RESULT    = f"{TD_DB}.original_label_predictions_fc"

def get_columns(con, db, table):
    sql = f"""
    SELECT ColumnName
    FROM DBC.ColumnsV
    WHERE DatabaseName = '{db}'
      AND TableName    = '{table.split('.')[-1]}'
    ORDER BY ColumnId
    """
    return pd.read_sql(sql, con)["ColumnName"].tolist()

def pick_ref_id(colnames):
    for c in ["gpc_id", "label_id", "orig_label_id", "id", "row_id"]:
        if c in colnames:
            return c
    raise ValueError(f"No suitable ID column in {REF_EMB}. Found: {colnames}")

with teradatasql.connect(host=TD_HOST, user=TD_USER, password=TD_PASS) as con:
    cur = con.cursor()

    item_cols = get_columns(con, TD_DB, ITEM_EMB)
    ref_cols  = get_columns(con, TD_DB, REF_EMB)

    if "row_id" not in item_cols:
        raise ValueError(f"{ITEM_EMB} must contain 'row_id'. Found: {item_cols}")

    ref_id_col = pick_ref_id(ref_cols)  # could be 'row_id' in your case
    # output alias for the reference id — force a DIFFERENT name than 'row_id'
    ref_id_out = "orig_label_id"               # choose what you like: 'gpc_id'/'orig_label_id'/...

    # ensure shared v* features & consistent order
    item_feats = [c for c in item_cols if c.startswith("v")]
    ref_feats  = [c for c in ref_cols  if c.startswith("v")]
    feats      = sorted(set(item_feats).intersection(ref_feats),
                        key=lambda x: int(x[1:]) if x[1:].isdigit() else x)
    if not feats:
        raise ValueError("No shared v* feature columns across both tables.")

    vec_cols         = ", ".join(feats)
    vec_cols_quoted  = ", ".join(f"'{c}'" for c in feats)

    # (re)create result table – output columns: row_id, gpc_id, score
    try:
        cur.execute(f"DROP TABLE {RESULT};")
    except Exception:
        pass

    create_sql = f"""
    CREATE MULTISET TABLE {RESULT} AS
    (
      SELECT
        o.Target_ID    AS row_id,
        o.Reference_ID AS {ref_id_out},
        1 - o.Distance AS score
      FROM TD_SYSFNLIB.TD_VectorDistance
      (
        ON (SELECT row_id, {vec_cols} FROM {ITEM_EMB}) AS TargetTable
        ON (SELECT {ref_id_col}, {vec_cols} FROM {REF_EMB})  AS ReferenceTable DIMENSION
        USING
          TargetIDColumn       ('row_id')
          RefIDColumn          ('{ref_id_col}')
          TargetFeatureColumns ({vec_cols_quoted})
          RefFeatureColumns    ({vec_cols_quoted})
          DistanceMeasure      ('cosine')
      ) AS o
      QUALIFY ROW_NUMBER() OVER (PARTITION BY o.Target_ID ORDER BY o.Distance) = 1
    ) WITH DATA
    PRIMARY INDEX (row_id);
    """
    cur.execute(create_sql)

print(f"✅ Rebuilt {RESULT} with top‑1 cosine similarity. Columns: row_id, {ref_id_out}, score")


*eval*

In [ ]:
import teradatasql
import pandas as pd

TD_HOST="iteration7-w9og53takluu3v27.env.clearscape.teradata.com"
TD_USER="demo_user"
TD_PASS="n8888888"

LABELS_CLAUSE = """
Labels(
 'Condiments, Dressings & Marinades','Furniture','Personal care, skin & body care','null',
 'Tea, Coffee & Hot Drinks','Sweets & Desserts','Hair, Shower, Bath & Soap','Fruits',
 'Nuts, Dates & Dried Fruits','Vegetables & Fruits','Home Appliances',
 'Sauces, Dressings & Condiments','Baby Care','Tea and Coffee','Disposables & Napkins',
 'Tins, Jars & Packets','Chips & Crackers','Soft Drinks & Juices','Cooking Ingredients',
 'Dairy & Eggs','Bakery','Vegetables & Herbs','Biscuits & Cakes','Candles & Air Fresheners',
 'Water','Rice, Pasta & Pulses','Poultry','Beef & Processed Meat','Home Textile',
 'Cleaning Supplies','Beef & Lamb Meat','Chocolates, Sweets & Desserts','Jams, Spreads & Syrups'
)
"""

with teradatasql.connect(host=TD_HOST, user=TD_USER, password=TD_PASS) as con:
    cur = con.cursor()

    # 0) (Optional) sanity probe of column names
    probe = """
    SEL TableName, ColumnId, ColumnName
    FROM DBC.ColumnsV
    WHERE DatabaseName='demo_user'
      AND TableName IN ('cleaned_final_with_id','original_labels_lookup','original_label_predictions_fc')
    ORDER BY TableName, ColumnId;
    """
    print(pd.read_sql(probe, con))

    # 1) Drop & create results table explicitly (avoid CTAS parser edge cases)
    try:
        cur.execute("DROP TABLE demo_user.results;")
    except Exception as e:
        if "3807" not in str(e):
            raise

    cur.execute("""
    CREATE MULTISET TABLE demo_user.results
    (
      actual_class    VARCHAR(512),
      predicted_class VARCHAR(512)
    );
    """)
    print("✅ Created demo_user.results")

    # 2) Populate results via INSERT ... SELECT
    insert_sql = """
    INSERT INTO demo_user.results (actual_class, predicted_class)
    SELECT
        cf."class"      AS actual_class,
        lkp."class"     AS predicted_class
    FROM demo_user.original_label_predictions_fc p
    JOIN demo_user.cleaned_final_with_id cf
        ON cf.row_id = p.row_id
    JOIN demo_user.original_labels_lookup lkp
        ON lkp.row_id = p.gpc_id;   -- your lookup uses 'row_id' as the label id
    """
    cur.execute(insert_sql)
    print(pd.read_sql("SEL COUNT(*) AS n FROM demo_user.results;", con))

    # 3) Persist evaluator output via CTAS (no VOLATILE, no OUT TABLE syntax)
    try:
        cur.execute("DROP TABLE demo_user.classification_metrics;")
    except Exception as e:
        if "3807" not in str(e):
            raise

    eval_ctas = f"""
    CREATE MULTISET TABLE demo_user.classification_metrics AS
    (
      SELECT *
      FROM TD_ClassificationEvaluator (
         ON demo_user.results AS InputTable
         USING
             ObservationColumn('actual_class')
             PredictionColumn('predicted_class')
             {LABELS_CLAUSE}
      ) AS dt
    ) WITH DATA;
    """
    cur.execute(eval_ctas)
    print("✅ Created demo_user.classification_metrics")

    # 4) Fetch metrics (persistent table)
    df = pd.read_sql("SELECT * FROM demo_user.classification_metrics;", con)

print("✅ Columns:", list(df.columns))

# Optional: sort by best-guess column names if present
metric_cols = ["metric_name", "Metric", "metric"]
label_cols  = ["class_label", "ClassLabel", "label", "Label"]

metric_col = next((c for c in metric_cols if c in df.columns), None)
label_col  = next((c for c in label_cols  if c in df.columns), None)

if metric_col and label_col:
    df = df.sort_values([metric_col, label_col])
elif metric_col:
    df = df.sort_values([metric_col])

print("\n🔎 Head:")
print(df.head(20))